# `dpac` — demonstration notebook

**Empirical companion to the preprint** *A declared percentage-based author-contribution system for weighted research metrics: a proposal* (Samaras, 2026).

This notebook does two things:

1. **Template (top of the notebook).** Drop in three to five ORCIDs and run all cells. The notebook will pull each researcher's publication record from OpenAlex, apply two algorithmic imputation schemes (equal share and harmonic counting) as stand-ins for the declared shares the proposal specifies, and produce both per-researcher comparison tables and a cross-researcher table that mirrors Table 1 of the preprint.
2. **Demonstration (bottom of the notebook).** A worked example on constructed mock data — six profiles matching the preprint archetypes A through F — so that the table machinery can be inspected end to end without any network call.

### Important caveat

Declared shares are not yet available in any bibliometric database. The equal-share and harmonic-counting schemes used below are *stand-ins* for the declared shares that the proposed system specifies. Equal share is field-agnostic but blunt; harmonic counting presumes a contribution-ordered byline convention, which holds in many fields but not in those that use alphabetical or non-contribution-ordered listings. Numbers from either scheme are not substitutes for declared-share metrics.


## 1. Setup


In [ ]:
# Standard imports plus dpac.
import os
import statistics

import matplotlib.pyplot as plt
import pandas as pd

from dpac import compute_profile
from dpac.openalex import Work


### OpenAlex API key

OpenAlex meters usage per day and identifies users by API key, free at https://openalex.org/settings/api; without a key the daily allowance is much smaller. Set `OPENALEX_API_KEY` in the shell before launching Jupyter: the package reads it from there and never prints it. Avoid typing the key into the notebook itself, so that it is not saved with the file.

The `mailto` contact of earlier versions is still passed on, but OpenAlex has ignored it since its 2026 switch to API keys.

In [ ]:
# Optional contact address (ignored by OpenAlex since 2026; kept for compatibility).
# os.environ['DPAC_MAILTO'] = 'your.email@example.org'
MAILTO = os.environ.get('DPAC_MAILTO')
print('OpenAlex API key:', 'set' if os.environ.get('OPENALEX_API_KEY') else 'not set (small daily allowance)')

## 2. Sample an anonymised cohort from OpenAlex (stratified)

The notebook draws a stratified random sample of authors from a defined OpenAlex pool, identifies them only as `Profile 1`, `Profile 2`, ..., and feeds those into the same machinery. This is the empirical companion the preprint refers to in §5.2: a small set of anonymised OpenAlex profiles, drawn from a defined field, with the two-tier computation applied under the two imputation schemes.

**Why stratified.** A flat random sample of authors drawn from a sample of works over-represents prolific researchers, because prolific researchers appear on more works in any random sample by construction. Stratified sampling slots authors into `works_count` bands and draws the same number from each band, producing a cohort that spans the activity spectrum rather than being dominated by the highest-output end.

**How the sampling works.** OpenAlex deprecated its concepts taxonomy in favour of topics, and topics-based filtering is available on the `/works` endpoint but not on `/authors`. The sampler takes a two-stage approach: it first samples works in the chosen topic (with `has_orcid:true`), then harvests the ORCID-bearing authors of those works and looks them up on `/authors` to apply activity thresholds; it then slots them into bands and draws `per_band` authors at random from each.

**Available pre-defined fields** (keys of `dpac.TOPICS`): `ocean_engineering`, `ocean_waves` (alias `coastal_engineering`), `wave_energy`, `hydraulic_engineering`, `oceanography`, `underwater_vehicles`. For any other field, look up the topic id at https://api.openalex.org/topics?search=<term> and pass the raw `T...` id instead.

**Anonymisation.** ORCIDs are stored internally in the `sampled` list but are never printed in the notebook output. The cohort is referenced as `Profile 1`, `Profile 2`, ... throughout.


In [ ]:
from dpac import sample_authors_stratified, TOPICS

# Configuration ---------------------------------------------------------
FIELD       = 'ocean_engineering'     # key from dpac.TOPICS, or a raw 'T...' id
BANDS       = [(5, 20), (21, 50), (51, 75), (76, 100), (101, 1000)]
PER_BAND    = 20                      # number of researchers per band (20 x 5 bands = 100)
MIN_H_INDEX = 5                       # h-index floor across all bands
SAMPLE_SEED = 20260522                # for reproducibility
WORKS_POOL  = 500                     # pool size at the works stage
SCHEMES     = ['equal', 'harmonic']   # imputation schemes for section 3
WORK_TYPES  = ['article']             # restrict to journal articles (set to None for all types)
# -----------------------------------------------------------------------

sampled = sample_authors_stratified(
    topic_id=FIELD,
    bands=BANDS,
    per_band=PER_BAND,
    min_h_index=MIN_H_INDEX,
    seed=SAMPLE_SEED,
    works_sample_size=WORKS_POOL,
    mailto=MAILTO,
)
print(f'Sampled {len(sampled)} authors from topic {FIELD!r} (TOPICS[{FIELD!r}] = {TOPICS.get(FIELD, FIELD)}):')
print(f'{"Profile":<14} {"works_count":>12} {"h_index":>9}   band')
for i, a in enumerate(sampled, 1):
    band = a.get('band', ('-', '-'))
    print(f'  Profile {i:>2}    {a["works_count"]:>9}       {a["h_index"]:>6}   {band[0]}-{band[1]}')

# Build the anonymised RESEARCHERS list that the rest of the notebook uses.
# ORCIDs are kept inside the `sampled` list but never printed.
RESEARCHERS = [(f'Profile {i}', a['orcid']) for i, a in enumerate(sampled, 1)]


### 2a. Optional — fall back to manual ORCIDs

If a hand-picked set of ORCIDs is preferred over the stratified sample, the cell below provides the alternative path. **The cell is fully commented by default — running it as-is does nothing.** To use it:

1. Uncomment the lines (select the block and press `Ctrl+/` in VS Code).
2. Replace the placeholder ORCIDs with real ones.
3. Run this cell. It will overwrite the `RESEARCHERS` list produced by section 2.

Note that with manual entry the *anonymisation by construction* of section 2 is lost — the labels become free-text and the ORCIDs are visible in this cell. Decide accordingly.


In [ ]:
# ----- Manual ORCID entry (alternative to the stratified sample above) -----
# Uncomment and edit to use; running as-is does nothing.

# RESEARCHERS = [
#     ('Profile 1', '0000-0000-0000-0000'),
#     ('Profile 2', '0000-0000-0000-0000'),
#     ('Profile 3', '0000-0000-0000-0000'),
# ]
# SCHEMES = ['equal', 'harmonic']      # already set by section 2; re-set here if needed
# WORK_TYPES = ['article']             # already set by section 2; re-set here if needed
# print(f'Manual cohort: {len(RESEARCHERS)} researchers loaded.')


## 3. Pull each ORCID and compute the metrics

The block below runs only when the `RESEARCHERS` list is non-empty. When the list is empty (the default state of this notebook), the block is skipped and the demonstration in section 6 takes over.


In [ ]:
results = {}
if RESEARCHERS:
    for label, orcid in RESEARCHERS:
        print(f'Fetching {label} ...')
        r = compute_profile(
            orcid=orcid,
            schemes=SCHEMES,
            mailto=MAILTO,
            work_types=WORK_TYPES,
        )
        results[label] = r
        print(f'  -> {r.paper_level.n_papers} works, {r.paper_level.total_citations} citations, raw h={r.paper_level.raw_h_index}')
else:
    print('No ORCIDs supplied. The mock-data demonstration in section 6 will run.')


## 4. Per-researcher three-column comparison

For each researcher, the table shows the paper-level set alongside the author-level set under each scheme.


In [ ]:
def per_researcher_table(label, result):
    '''Return a tidy DataFrame for one researcher.'''
    rows = []
    p = result.paper_level
    rows.append({
        'metric': 'Papers',
        'paper_level': p.n_papers,
        **{f'author_level_{s}': round(result.author_level[s].effective_papers, 2)
           for s in result.author_level},
    })
    rows.append({
        'metric': 'Total citations',
        'paper_level': p.total_citations,
        **{f'author_level_{s}': round(result.author_level[s].weighted_citations, 1)
           for s in result.author_level},
    })
    rows.append({
        'metric': 'h-index (raw / fractional)',
        'paper_level': p.raw_h_index,
        **{f'author_level_{s}': round(result.author_level[s].fractional_h_index, 2)
           for s in result.author_level},
    })
    df = pd.DataFrame(rows).set_index('metric')
    df.attrs['label'] = label
    return df

for label, r in results.items():
    print(f'\n=== {label} ===')
    print(per_researcher_table(label, r))


## 5. Cross-researcher comparison — Table 1 analogue

One row per researcher; columns mirror Table 1 of the preprint.


In [ ]:
def cross_researcher_table(results_dict):
    '''Build the Table-1-analogue across multiple researchers.'''
    rows = []
    for label, r in results_dict.items():
        row = {
            'Researcher': label,
            'Papers': r.paper_level.n_papers,
            'Total citations': r.paper_level.total_citations,
            'Raw h-index': r.paper_level.raw_h_index,
        }
        for s, m in r.author_level.items():
            row[f'Weighted citations ({s})'] = round(m.weighted_citations, 1)
            row[f'Fractional h ({s})'] = round(m.fractional_h_index, 2)
        rows.append(row)
    return pd.DataFrame(rows).set_index('Researcher')

if results:
    cross = cross_researcher_table(results)
    display(cross)
else:
    print('Section 5 will populate when RESEARCHERS is non-empty.')


## 6. Demonstration on mock data — the six preprint archetypes

This section runs unconditionally so the table machinery can be inspected end to end without any network call. The six mock profiles are constructed to match, in author count and citation pattern, the six archetypes of §4 of the preprint:

- **A** — solo and lead-author specialist (~15 papers, small groups, lead position).
- **B** — small-group equal collaborator (~20 papers, 2–3 authors, mixed position).
- **C** — balanced mid-size collaborator (~30 papers, 4–7 authors).
- **D** — supervisory contributor (~80 papers, mostly last-author, 4–10 authors).
- **E** — consortium contributor (~60 papers, mix of consortium and small-group).
- **F** — honorary/gift author (~50 papers, middle positions across small/medium groups).

The mock data is intentionally noisy within each archetype, so the resulting table is illustrative rather than exact. The expected qualitative behaviour — small paper-to-author-level divergence for A; large divergence for E and F — is what the demonstration is meant to verify.


In [ ]:
import random

RNG = random.Random(20260522)  # deterministic for reproducibility

def make_work(cit, n, pos, year, idx, focal_known=True):
    return Work(
        openalex_id=f'MOCK-{idx}',
        doi=None,
        title=f'mock work {idx}',
        publication_year=year,
        cited_by_count=cit,
        n_authors=n,
        focal_position=pos,
        focal_position_known=focal_known,
    )

def gen_profile_A(start_idx=0):
    '''Solo / lead-author specialist: 15 papers, mostly first or sole author.'''
    works = []
    for k in range(15):
        n = RNG.choice([1, 2, 2, 3])
        pos = 1  # always lead
        # Citations: solo papers get fewer; modest range overall.
        cit = max(0, int(RNG.gauss(40, 18)))
        year = 2008 + k          # one paper per year
        works.append(make_work(cit, n, pos, year, start_idx + k))
    return works

def gen_profile_B(start_idx=100):
    '''Small-group equal collaborator: 20 papers, 2-3 authors.'''
    works = []
    for k in range(20):
        n = RNG.choice([2, 2, 3, 3, 3])
        pos = RNG.choice(list(range(1, n + 1)))  # mixed
        cit = max(0, int(RNG.gauss(40, 15)))
        year = 2008 + (k * 12) // 20
        works.append(make_work(cit, n, pos, year, start_idx + k))
    return works

def gen_profile_C(start_idx=200):
    '''Balanced mid-size collaborator: 30 papers, 4-7 authors.'''
    works = []
    for k in range(30):
        n = RNG.choice([4, 4, 5, 5, 6, 7])
        pos = RNG.choice(list(range(1, n + 1)))
        cit = max(0, int(RNG.gauss(50, 20)))
        year = 2008 + (k * 15) // 30
        works.append(make_work(cit, n, pos, year, start_idx + k))
    return works

def gen_profile_D(start_idx=300):
    '''Supervisory: 80 papers, mostly last-author, 4-10 authors.'''
    works = []
    for k in range(80):
        n = RNG.choice([4, 5, 6, 7, 8, 9, 10])
        # 70% last-author, 20% middle, 10% first
        r = RNG.random()
        if r < 0.7:
            pos = n
        elif r < 0.9:
            pos = RNG.randint(2, max(2, n - 1))
        else:
            pos = 1
        cit = max(0, int(RNG.gauss(50, 25)))
        year = 2000 + (k * 24) // 80   # long active period
        works.append(make_work(cit, n, pos, year, start_idx + k))
    return works

def gen_profile_E(start_idx=400):
    '''Consortium: 60 papers, 50 consortium + 10 small-group.'''
    works = []
    for k in range(50):
        # Consortium: 10 to 100+ co-authors
        n = RNG.choice([15, 25, 40, 60, 100, 150])
        # Position drawn anywhere on the list
        pos = RNG.randint(2, max(2, n - 1))  # not first/last on consortium
        cit = max(0, int(RNG.gauss(120, 60)))  # consortium papers cited heavily
        year = 2010 + (k * 14) // 50
        works.append(make_work(cit, n, pos, year, start_idx + k))
    for k in range(10):
        n = RNG.choice([3, 4, 5])
        pos = RNG.choice(list(range(1, n + 1)))
        cit = max(0, int(RNG.gauss(30, 15)))
        year = 2010 + (k * 14) // 10
        works.append(make_work(cit, n, pos, year, start_idx + 50 + k))
    return works

def gen_profile_F(start_idx=500):
    '''Honorary / gift author: 50 papers, persistent middle position on small-medium groups.'''
    works = []
    for k in range(50):
        n = RNG.choice([3, 4, 5, 6, 8, 10, 12, 15])
        # Always middle (decoupled from lead role)
        pos = RNG.randint(2, max(2, n - 1))
        cit = max(0, int(RNG.gauss(55, 22)))
        year = 2005 + (k * 19) // 50
        works.append(make_work(cit, n, pos, year, start_idx + k))
    return works

mock_profiles = {
    'A — Solo / lead-author specialist': gen_profile_A(),
    'B — Small-group equal collaborator': gen_profile_B(),
    'C — Balanced mid-size collaborator': gen_profile_C(),
    'D — Supervisory contributor': gen_profile_D(),
    'E — Consortium contributor': gen_profile_E(),
    'F — Honorary / gift author': gen_profile_F(),
}

for label, works in mock_profiles.items():
    print(f'{label}: {len(works)} works, total citations = {sum(w.cited_by_count for w in works)}')


### 6.1 — Run the comparison on the mock data


In [ ]:
mock_results = {}
for label, works in mock_profiles.items():
    r = compute_profile(works=works, schemes=SCHEMES)
    mock_results[label] = r

mock_table = cross_researcher_table(mock_results)
display(mock_table)


### 6.2 — Sanity check against Table 1 of the preprint, and a key caveat

The preprint quotes order-of-magnitude estimates for each archetype. The mock-data run is constructed to land in the same ballpark on paper-level totals (citation counts, paper counts, raw *h*) and on the broad qualitative shape of the author/paper ratio across archetypes: smallest divergence for Profile A, severe collapse on the consortium component of Profile E.

**One archetype highlights a fundamental limitation of any algorithmic imputation:** Profile F (honorary / gift author). The preprint's Profile F has *declared* shares persistently around 5–10%, producing an author-level total at roughly 8% of the paper-level total. Under *equal-share imputation*, however, Profile F's middle-position contributions on small-to-medium groups receive shares of around 1/*n* ∈ [0.07, 0.33] — the imputation cannot distinguish an honorary author from a genuine equal collaborator on the same byline. Under *harmonic-counting imputation*, the honorary author's middle-position penalty drives the share lower, but the gap to a genuine declared share of 5–10% remains.

This is not a bug in the imputation; it is the substantive case for the proposal. **Imputation schemes cannot, by construction, detect contribution patterns that diverge from the byline.** The author/paper ratio under either scheme is therefore an *upper bound* on what a declared-share computation would yield for a researcher whose actual shares fall below the byline-implied level. The preprint Table 1 numbers for Profile F should be read against this caveat: the mock-data row for F will *not* reproduce the ~8% author/paper ratio under either equal-share or harmonic counting, because the byline-position information available to OpenAlex does not carry the honorary-authorship signal.

The exact numbers depend on the random seed (set to `20260522` for reproducibility); the qualitative ordering is what matters here.


In [ ]:
# Add the 'author-as-fraction-of-paper-level' diagnostic, since the
# direction of divergence is the substantive content.
mock_table_with_ratio = mock_table.copy()
for scheme in SCHEMES:
    col = f'Weighted citations ({scheme})'
    ratio_col = f'Author / paper ratio ({scheme})'
    mock_table_with_ratio[ratio_col] = (
        mock_table_with_ratio[col] / mock_table_with_ratio['Total citations']
    ).round(3)
display(mock_table_with_ratio[
    ['Total citations', 'Raw h-index']
    + [f'Weighted citations ({s})' for s in SCHEMES]
    + [f'Fractional h ({s})' for s in SCHEMES]
    + [f'Author / paper ratio ({s})' for s in SCHEMES]
])


### 6.3 — Real cohort vs mock-data archetypes

Side-by-side comparison of the real OpenAlex cohort against the six mock-data archetypes from sections 6.1 and 6.2. The intent is *not* to expect the real cohort to match the archetypes — the archetypes are illustrative (§4 of the preprint) and the real cohort is a stratified random sample of the field. The interesting empirical question is the opposite: **where on the contribution-style spectrum (which the archetypes span) do real researchers in this field actually cluster?**

The comparison metric below is the author/paper citation ratio under each imputation scheme — Σ (c_i · m_p) / Σ m_p across the researcher's record. The ratio is bounded in [0, 1]. Higher values mean the author's declared (here: imputed) share captures most of the paper-level total; lower values mean the imputed share scales the paper-level total substantially downward.

The mock-data archetypes give a reference frame: A ≈ 0.6–0.8 (consistent first/sole author), B ≈ 0.3–0.4 (small-group equal), C ≈ 0.15–0.25 (mid-size collaborator), D ≈ 0.10–0.20 (supervisory contributor), E ≈ 0.03–0.05 (consortium contributor), F ≈ 0.10–0.20 (honorary, *as imputation can detect — true F is lower under declared shares*).


In [ ]:
# Build a unified table: each row is one researcher (real or mock),
# columns are the author/paper ratio under each scheme.
if not results:
    print('No real cohort to compare against. Run sections 2 and 3 first.')
else:
    rows = []
    for label, r in results.items():
        p_total = max(r.paper_level.total_citations, 1)
        rows.append({
            'who': label,
            'kind': 'real',
            'papers': r.paper_level.n_papers,
            'paper_cit': r.paper_level.total_citations,
            'ratio_equal':   round(r.author_level['equal'].weighted_citations / p_total, 3),
            'ratio_harmonic': round(r.author_level['harmonic'].weighted_citations / p_total, 3),
        })
    for label, r in mock_results.items():
        p_total = max(r.paper_level.total_citations, 1)
        rows.append({
            'who': label,
            'kind': 'mock',
            'papers': r.paper_level.n_papers,
            'paper_cit': r.paper_level.total_citations,
            'ratio_equal':   round(r.author_level['equal'].weighted_citations / p_total, 3),
            'ratio_harmonic': round(r.author_level['harmonic'].weighted_citations / p_total, 3),
        })
    combined = pd.DataFrame(rows).set_index('who')
    display(combined)


### Visualisation — distribution of ratios

The plot below places each real-cohort researcher and each mock archetype on the same x-axis (paper-level citation total) and y-axis (author/paper ratio under harmonic counting). Mock archetypes are shown as labelled markers; real cohort members as unlabelled markers. The intended reading: do the real cohort members cluster around any one archetype, or are they spread across the spectrum?


In [ ]:
# Save outputs flag — set to False to suppress writing files.
SAVE_OUTPUTS = True
OUTPUT_STEM  = 'cohort_ocean_engineering'

if results:
    fig, (ax_eq, ax_ha) = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
    real_df = combined[combined['kind'] == 'real']
    mock_df = combined[combined['kind'] == 'mock']
    # Cohort markers: light teal-blue circles with dark-teal edge, matching
    # Figures 1 and 2 of the preprint (chat-11-addendum palette).
    # Archetype markers: orange diamonds with dark-orange edge, sized at
    # ~1.6x the cohort area so the letters read but the archetypes do not
    # dominate the cloud they sit within.
    for ax, scheme_col, scheme_label in [
        (ax_eq, 'ratio_equal',    'equal share'),
        (ax_ha, 'ratio_harmonic', 'harmonic counting'),
    ]:
        ax.scatter(real_df['paper_cit'], real_df[scheme_col],
                   s=38, alpha=0.85,
                   facecolor='#5BA3B5', edgecolor='#2D5F6E', linewidth=0.6,
                   label='Real cohort (anonymised)')
        ax.scatter(mock_df['paper_cit'], mock_df[scheme_col],
                   s=60, marker='D',
                   facecolor='#D88536', edgecolor='#7A4A1F', linewidth=0.7,
                   label='Mock archetypes')
        for label, row in mock_df.iterrows():
            archetype_letter = label.split(' \u2014 ')[0]
            ax.annotate(archetype_letter,
                        (row['paper_cit'], row[scheme_col]),
                        textcoords='offset points', xytext=(8, 4), fontsize=11,
                        fontweight='bold', color='#1F3A56')
        ax.set_xscale('log')
        ax.set_xlabel('Paper-level total citations (log scale)')
        ax.set_title(f'Author / paper ratio under {scheme_label}')
        ax.grid(True, alpha=0.3)
    ax_eq.set_ylabel('Author / paper ratio')
    ax_eq.legend(loc='best')
    fig.suptitle('Real cohort vs mock-data archetypes — '
                 'equal-share and harmonic-counting imputations',
                 y=1.00)
    plt.tight_layout()
    if SAVE_OUTPUTS:
        fig.savefig(f'{OUTPUT_STEM}.png', dpi=150, bbox_inches='tight')
        print(f'Saved figure to: {OUTPUT_STEM}.png')
    plt.show()


### Quick summary statistics on the real cohort


In [ ]:
if results:
    real_df = combined[combined['kind'] == 'real']
    print(f'Real cohort: {len(real_df)} researchers')
    print(f'\nAuthor/paper ratio (harmonic counting):')
    print(f'  median = {real_df["ratio_harmonic"].median():.3f}')
    print(f'  mean   = {real_df["ratio_harmonic"].mean():.3f}')
    print(f'  range  = [{real_df["ratio_harmonic"].min():.3f}, {real_df["ratio_harmonic"].max():.3f}]')
    print(f'\nAuthor/paper ratio (equal share):')
    print(f'  median = {real_df["ratio_equal"].median():.3f}')
    print(f'  mean   = {real_df["ratio_equal"].mean():.3f}')
    print(f'  range  = [{real_df["ratio_equal"].min():.3f}, {real_df["ratio_equal"].max():.3f}]')

    # Persist the combined table as CSV alongside the notebook, controlled by
    # the SAVE_OUTPUTS flag set at the top of the visualisation cell above.
    if SAVE_OUTPUTS:
        combined.to_csv(f'{OUTPUT_STEM}.csv')
        print(f'\nSaved combined table to: {OUTPUT_STEM}.csv')


## 7. Notes for the user

- **Cache.** OpenAlex responses cache to `~/.dpac_cache` by default. To force a refresh on a researcher, delete the cache file or pass `use_cache=False` to `compute_profile`.
- **Field-byline caveat for harmonic counting.** If any researcher in the comparison set works in a field with alphabetical bylines, the harmonic-counting column should be read with that qualifier. The equal-share column is field-agnostic.
- **Byline order and long author lists.** The harmonic share uses the focal author's exact byline position, read from the order of OpenAlex's `authorships` array. OpenAlex lists at most 100 authors per work: works with a 100-name list are flagged (`authors_capped`) and their equal and harmonic shares are approximate (upper bounds), and works on which the focal author comes after the 100th name cannot be matched by ORCID and are skipped.
- **Coverage.** A researcher with a partially-populated ORCID on OpenAlex will appear to have fewer publications than they do. The OpenAlex profile page (`https://openalex.org/orcid/NNNN-NNNN-NNNN-NNNN`) is the appropriate sanity check before drawing conclusions.

## 8. Citation

If results from this notebook are used in published work, please cite the preprint:

> Samaras, A. G. (2026). A declared percentage-based author-contribution system for weighted research metrics: a proposal. *MetaArXiv preprint*. https://doi.org/10.31222/osf.io/3mkz2_v1
